
# HEFT Baseline — Conditional Penalty (Only for **Inter-level** comm via **RID/RTSN**)

This version matches your rule precisely:

- Compute nodes: IDs starting with **E** (Edge), **F** (Fog), **P** (Cloud).
- **Level detection**
  - `E*` → level **E**
  - `F*` → level **F**
  - `P<d>*` → level **C<d>** (e.g., `P1xxx → C1`, `P2xxx → C2`, `P3xxx → C3`). If no digit, falls back to **P**.
- **Penalty routers**: only nodes whose ID starts with `RID` or `RTSN`.
- **Penalty condition**: apply ×10 for **each** RID/RTSN present on the shortest-latency path **only if the sender and receiver are on different levels** (e.g., E↔F, E↔C1, F↔C2, C1↔C3, etc.).  
  If they are on the **same level** (E–E, F–F, C1–C1, C2–C2, C3–C3), **no penalty** is applied even if path contains routers.
- Communication model:
  \[
  t_{comm}(p\to q, S) \;=\; \big(\sum L_e \;+\; S \cdot \sum \tfrac{1}{B_e}\big) \times \text{penalty}
  \]
  where `penalty = 10^{#RID/RTSN_on_path}` **iff** `level(p) != level(q)`, otherwise `penalty = 1`.
- HEFT: upward ranks + EFT placement, ignoring `can_run_on`.

> Default model path: `/mnt/data/TC100_NewPM.json`. Adjust the path below if needed.


In [1]:

import json, time, math, re
from collections import defaultdict, deque
from math import inf
from pathlib import Path

import pandas as pd


In [2]:

# ---- Node & level classification ----
def is_compute(node_id: str) -> bool:
    nid = node_id.upper()
    return nid.startswith("E") or nid.startswith("F") or nid.startswith("P")

def is_penalty_router(node_id: str) -> bool:
    nid = node_id.upper()
    return nid.startswith("RID") or nid.startswith("RTSN")

def node_level(node_id: str) -> str:
    """Map compute IDs to levels: E, F, C1, C2, C3 (or P as fallback).
    Non-compute nodes return their own ID tag (unused for penalty check)."""
    nid = node_id.upper()
    if nid.startswith("E"):
        return "E"
    if nid.startswith("F"):
        return "F"
    if nid.startswith("P"):
        # Try to extract the first digit after P to decide C1/C2/C3
        m = re.match(r"P(\d)", nid)
        if m:
            return f"C{m.group(1)}"
        return "P"
    return nid  # routers etc.; not used for level comparison


In [3]:

# --- Configure your model path here ---
JSON_PATH = "C:/Users/oheka/Desktop/FTCodes/ModularFTcodes/Platforms/TC100_NewPM.json"
# JSON_PATH = 'C:/Users/oheka/Desktop/FTCodes/ModularFTcodes/Platforms/Omar/Omar/250/Application/TC250.json'
with open(JSON_PATH, "r") as f:
    data = json.load(f)

application = data["application"]
platform_json = data["platform"]

print("Loaded:", JSON_PATH)
print("Jobs:", len(application.get("jobs", [])), "Messages:", len(application.get("messages", [])))
print("Platform nodes/links:", len(platform_json.get("nodes", [])), len(platform_json.get("links", [])))


Loaded: C:/Users/oheka/Desktop/FTCodes/ModularFTcodes/Platforms/TC100_NewPM.json
Jobs: 100 Messages: 212
Platform nodes/links: 195 574


In [4]:

# ---- Build application DAG ----
tasks = {j["id"] for j in application["jobs"]}
comp_time = {j["id"]: float(j["processing_times"]) for j in application["jobs"]}

parents = {t: set() for t in tasks}
children = {t: set() for t in tasks}
msg_size = {}

for m in application.get("messages", []):
    u, v = m["sender"], m["receiver"]
    size = float(m.get("size", 0.0))
    if u not in tasks or v not in tasks:
        raise ValueError(f"Message references unknown task: {u}->{v}")
    children[u].add(v)
    parents[v].add(u)
    msg_size[(u, v)] = size

# Topological check
indeg = {u: len(parents[u]) for u in parents}
q = deque([u for u in indeg if indeg[u] == 0])
order = []
while q:
    u = q.popleft()
    order.append(u)
    for v in children[u]:
        indeg[v] -= 1
        if indeg[v] == 0:
            q.append(v)

if len(order) != len(indeg):
    raise ValueError("The application graph is not a DAG.")

print("Task DAG OK. Topological size:", len(order))


Task DAG OK. Topological size: 100


In [5]:

# ---- Build platform graph (undirected, weighted) ----
DEFAULT_LINK_LATENCY = 1.0
DEFAULT_BANDWIDTH = 1.0

nodes = set()
adj = defaultdict(list)  # node -> list[(neighbor, latency, bandwidth)]

for n in platform_json.get("nodes", []):
    nodes.add(n["id"])

for link in platform_json.get("links", []):
    a = link["start"]
    b = link["end"]
    L = float(link.get("latency", DEFAULT_LINK_LATENCY))
    B = float(link.get("bandwidth", DEFAULT_BANDWIDTH))
    nodes.add(a); nodes.add(b)
    adj[a].append((b, L, B))
    adj[b].append((a, L, B))

compute_nodes = sorted([n for n in nodes if is_compute(n)])
if not compute_nodes:
    raise ValueError("No compute nodes found (E*/F*/P*). Please check your platform definition.")

print("Compute nodes:", len(compute_nodes))


Compute nodes: 94


In [6]:

# ---- Dijkstra: minimize sum of latency ----
def shortest_latency_path(src: str, dst: str):
    if src == dst:
        return [src], 0.0, []  # path, sum_latency, list_of_bandwidths
    dist = {n: float("inf") for n in nodes}
    prev = {n: None for n in nodes}
    prev_edge_B = {n: None for n in nodes}
    dist[src] = 0.0
    visited = set()
    while True:
        u = None
        best = float("inf")
        for n in nodes:
            if n not in visited and dist[n] < best:
                best = dist[n]; u = n
        if u is None or u == dst:
            break
        visited.add(u)
        for v, L, B in adj.get(u, []):
            alt = dist[u] + L
            if alt < dist[v]:
                dist[v] = alt
                prev[v] = u
                prev_edge_B[v] = B

    if math.isinf(dist[dst]):
        # disconnected; return pseudo path to keep going
        return [src, dst], 1e9, [1.0]

    # Reconstruct path and bandwidths
    path = []
    B_list = []
    cur = dst
    while cur is not None:
        path.append(cur)
        if prev[cur] is not None:
            B_list.append(prev_edge_B[cur])
        cur = prev[cur]
    path.reverse()
    B_list.reverse()
    return path, dist[dst], B_list


In [7]:

# ---- Precompute pairwise stats between compute nodes ----
pair_stats = {}  # (p,q) -> dict(sum_latency, sum_inv_bw, rid_rtsn_count)
start_pre = time.time()
for p in compute_nodes:
    for q in compute_nodes:
        if (p, q) in pair_stats:
            continue
        path, sum_lat, B_list = shortest_latency_path(p, q)
        sum_inv_bw = sum(1.0 / (b if (b and b > 0) else DEFAULT_BANDWIDTH) for b in B_list)
        rid_rtsn_count = sum(1 for n in path if is_penalty_router(n))
        d = {"sum_latency": sum_lat, "sum_inv_bw": sum_inv_bw, "rid_rtsn_count": rid_rtsn_count}
        pair_stats[(p, q)] = d
        pair_stats[(q, p)] = d  # symmetric for undirected graph
pre_time = time.time() - start_pre
print(f"Pairwise precompute done in {pre_time:.3f}s for {len(compute_nodes)} nodes.")


Pairwise precompute done in 6.390s for 94 nodes.


In [8]:

def comm_time(p: str, q: str, size: float) -> float:
    if p == q:
        return 0.0
    st = pair_stats[(p, q)]
    base = st["sum_latency"] + size * st["sum_inv_bw"]
    # Conditional penalty: apply only if levels differ
    lp, lq = node_level(p), node_level(q)
    if lp != lq:
        penalty = 15 ** st["rid_rtsn_count"]
    else:
        penalty = 1
    return base * penalty

def compute_upward_rank(tasks, children, comp_time, compute_nodes, msg_size):
    # Average comm for each edge = mean over all compute-node pairs
    all_pairs = [(p, q) for p in compute_nodes for q in compute_nodes]
    avg_comm = {}
    for i in tasks:
        for j in children[i]:
            size = msg_size.get((i, j), 0.0)
            if size <= 0:
                avg_comm[(i, j)] = 0.0
                continue
            total = 0.0
            for p, q in all_pairs:
                total += comm_time(p, q, size)
            avg_comm[(i, j)] = total / len(all_pairs)

    rank_u = {}
    def dfs(u):
        if u in rank_u:
            return rank_u[u]
        if not children[u]:
            rank_u[u] = comp_time[u]
        else:
            best = 0.0
            for v in children[u]:
                best = max(best, avg_comm[(u, v)] + dfs(v))
            rank_u[u] = comp_time[u] + best
        return rank_u[u]

    for t in tasks:
        dfs(t)
    return rank_u

def heft_schedule(tasks, parents, children, comp_time, compute_nodes, msg_size):
    rank_u = compute_upward_rank(tasks, children, comp_time, compute_nodes, msg_size)
    priority = sorted(tasks, key=lambda t: (-rank_u[t], t))

    avail = {p: 0.0 for p in compute_nodes}
    assign = {}
    schedule = {}

    for u in priority:
        best_proc, best_start, best_finish = None, None, float("inf")
        for p in compute_nodes:
            data_ready = 0.0
            for par in parents[u]:
                par_p = assign[par]
                size = msg_size.get((par, u), 0.0)
                data_ready = max(data_ready, schedule[par][1] + comm_time(par_p, p, size))
            start = max(avail[p], data_ready)
            finish = start + comp_time[u]
            if finish < best_finish:
                best_finish = finish
                best_start = start
                best_proc = p
        assign[u] = best_proc
        schedule[u] = (best_start, best_finish)
        avail[best_proc] = best_finish

    makespan = max(e for (_, e) in schedule.values())
    return schedule, assign, makespan, rank_u


In [9]:

t0 = time.time()
schedule, assign, makespan, rank_u = heft_schedule(tasks, parents, children, comp_time, compute_nodes, msg_size)
runtime = time.time() - t0

rows = []
for t, (s, e) in sorted(schedule.items(), key=lambda kv: kv[1][0]):
    rows.append({"task": t, "start": s, "end": e, "processor": assign[t], "rank_u": rank_u[t]})
df = pd.DataFrame(rows)
df.head(10)


,task,start,end,processor,rank_u
0,90,0.0,130.0,E31,46986.961521
1,76,0.0,78.0,E32,45884.301494
2,79,0.0,64.0,E33,44556.091444
3,95,0.0,51.0,E34,44383.981440
4,50,0.0,105.0,F101,43393.211408
5,63,0.0,145.0,F102,41537.781349
6,92,0.0,89.0,F103,40841.121322
7,53,0.0,125.0,F104,39910.461295
8,44,0.0,65.0,F105,39609.241286
9,67,0.0,128.0,F106,39365.021277


In [10]:

print("== HEFT finished ==")
print(f"Compute nodes ({len(compute_nodes)}): {', '.join(compute_nodes[:10])}{'...' if len(compute_nodes)>10 else ''}")
print(f"Makespan: {makespan:.3f}")
print(f"Runtime:  {runtime:.3f} s (pairwise precompute above: {pre_time:.3f} s)")


== HEFT finished ==
Compute nodes (94): E31, E32, E33, E34, F101, F102, F103, F104, F105, F106...
Makespan: 4329.000
Runtime:  5.054 s (pairwise precompute above: 6.390 s)


In [11]:

# Save artifacts next to your JSON file path
base_dir = Path(JSON_PATH).parent
csv_path = base_dir / "heft_schedule.csv"
json_path = base_dir / "heft_schedule.json"

df.to_csv(csv_path, index=False)
out_json = {
    "makespan": makespan,
    "runtime_seconds": runtime,
    "compute_nodes": compute_nodes,
    "schedule": {int(r["task"]): {"start": r["start"], "end": r["end"], "processor": r["processor"], "rank_u": r["rank_u"]}
                 for _, r in df.iterrows()}
}
with open(json_path, "w") as f:
    json.dump(out_json, f, indent=2)

print("Saved:")
print(" -", csv_path)
print(" -", json_path)


Saved:
 - C:\Users\oheka\Desktop\FTCodes\ModularFTcodes\Platforms\heft_schedule.csv
 - C:\Users\oheka\Desktop\FTCodes\ModularFTcodes\Platforms\heft_schedule.json
